# Структура
- DINOv2 для поиска пар изображений  (получение эмеддингов) + (сравнение расстояния эмбедигов)
- ALIKED и LightGlue для извлечения и сопоставления признаков
- COLMAP для 3D-реконструкции.

# Dino v2
- изображение -> тезор эмбеддингов (512) 
- строится distances - булевая матрица:
пусть есть 4 изображения, тогда:
```
[[False, True,  False, True ],  # Изображение 1 похоже на 2 и 4
 [True,  False, True,  False],  # Изображение 2 похоже на 1 и 3
 [False, True,  False, False],  # Изображение 3 похоже на 2
 [True,  False, False, False]]  # Изображение 4 похоже на 1 
```

In [ ]:
!pip install --no-index /kaggle/input/imc2024-packages-lightglue-rerun-kornia/* --no-deps
!pip install pycolmap
!pip install kornia
!pip install git+https://github.com/cvg/LightGlue.git

!apt-get install -y colmap

In [ ]:
import pandas as pd
import os
import sys
import shutil
import numpy as np
from scipy.spatial.transform import Rotation as R
import cv2
import torch
import pycolmap
import array
import h5py
import subprocess
import kornia as K
import kornia.feature as KF
import torchvision.transforms as T
import torch.nn.functional as F
from lightglue import ALIKED
from lightglue import LightGlue, match_pair
from lightglue.utils import load_image
from kornia.io import ImageLoadType
from transformers import AutoImageProcessor, AutoModel
sys.path.append('/kaggle/input/imc25-utils')
# Colmap импорты для работы с бд
from database import COLMAPDatabase
from h5_to_db import add_keypoints, add_matches, import_into_colmap


In [ ]:
MIN_PAIRS = 5  # Минимальное количество пар для изолированных изображений
TOLERANCE = 100  # Порог для исключения слишком далеких пар
MIN_MATCHES = 100 # максимальное расстояние между эмбеддингами

In [ ]:
data_path = "/kaggle/input/image-matching-challenge-2025"
test_path = os.path.join(data_path, "test")
train_path = os.path.join(data_path, "train")   # не используется

### image_id | dataset | scene | image | rotation_matix | translation_vector

In [ ]:
submission = pd.read_csv(os.path.join(data_path, "sample_submission.csv"))

### Определение пересечения изображений на основе эмбеддингов DinoV2

In [ ]:
def get_image_pairs(dataset, scene, group, device='cuda' if torch.cuda.is_available() else 'cpu'):
    # список путей к изображениям
    image_paths = [os.path.join(dataset, row['image']) for _, row in group.iterrows()]
    # dataframe для каждого изображения
    indices = [idx for idx, _ in group.iterrows()]

    # обработка крайних случаев
    if len(image_paths) < 2:
        return []
    if len(image_paths) <= 3:
        return list(combinations(range(len(image_paths)), 2))
    
    # загрузка модели DINOv2 в режиме оценки
    processor = AutoImageProcessor.from_pretrained('/kaggle/input/dinov2/pytorch/base/1/')
    model = AutoModel.from_pretrained('/kaggle/input/dinov2/pytorch/base/1/').eval().to(device)

    embeddings = []
    # выключение градиентов | 224x224 -> Tensor | получаем нормализованные эмбеддинги
    for img_path in image_paths:
        # (1, 3, H, W) - преобразование для Dino ( (batch_size, channels, height, width) )
        image = K.io.load_image(img_path, K.io.ImageLoadType.RGB32, device=device)[None, ...]
        with torch.inference_mode():
            inputs = processor(images=image, return_tensors="pt", do_rescale=False, 
                             do_resize=True, do_center_crop=True, size=224).to(device)
            outputs = model(**inputs)
            embedding = F.normalize(outputs.last_hidden_state.max(dim=1)[0])    # выполняет max-pooling по временному измерению, получая тензор формы (1, embedding_dim
            
        embeddings.append(embedding)
    

    # объединение список тензоров в один тензор по первому измерению
    embeddings = torch.cat(embeddings, dim=0)
    # евклидово расстояние между эмбеддингами в намай массив
    distances = torch.cdist(embeddings, embeddings).cpu().numpy()
    # булевый массив расстоний (без самопересечений)
    distances_ = distances <= 0.3
    np.fill_diagonal(distances_, False) 
    z = distances_.sum(axis=1)  # кол-во пар для каждого изображения
    idxs0 = np.where(z == 0)[0] # изолированные изображения
    # добавление MIN_PAIRS изображений к каждому изолированному изображения
    for idx0 in idxs0:
        t = np.argsort(distances[idx0])[1:MIN_PAIRS + 1]
        distances_[idx0, t] = True
    
    # исключение сильно далеких пар
    s = np.where(distances >= TOLERANCE)
    distances_[s] = False

    pairs = []
    for i in range(len(image_paths)):
        for j in range(len(image_paths)):
            if distances_[i][j]:
                if i < j:
                    pairs.append((indices[i], indices[j]))
                else:
                    pairs.append((indices[j], indices[i]))
    
    # массив без дубликатов с парами индексов изображений
    pairs = list(set(pairs))
    return pairs

### Что такое дескрипторы?
Дескрипторы — это числовые векторы, описывающие окрестности ключевых точек. Они нужны для сопоставления точек между изображениями (например, чтобы понять, что угол окна на одном изображении соответствует углу окна на другом).

In [ ]:
"""
- обрабатывает группу изображений, 
- извлекает признаки, сопоставляет их 
- выполняет реконструкцию с помощью COLMAP.
"""

def process_group(dataset, scene, group, indices, pairs):

    # создание рабочей директории для проекта
    project_dir = f"/kaggle/working/project_{dataset}_{scene}_{indices[0]}"
    os.makedirs(project_dir, exist_ok=True)
    db_path = os.path.join(project_dir, "database.db")
    
    # удаление старой бд
    if os.path.exists(db_path):
        os.remove(db_path)
    
    # копирование изображений из test директории в рабочую
    image_names = []
    test_path = os.path.join(data_path, "test")
    for _, row in group.iterrows(): # _ - номер строки | row = image_id, dataset, scene, image
        image_path = os.path.join(test_path, dataset, row['image']) # /kaggle/input/image-matching-challenge-2025/test/ETs/image1.png
        if os.path.exists(image_path):
            dest_path = os.path.join(project_dir, os.path.basename(row['image']))
            shutil.copy(image_path, dest_path)
            image_names.append(os.path.basename(row['image']))
    
    # если в группе датасета < 2 img:
    if len(image_names) < 2:
        print(f"Skipping group with insufficient images: {len(image_names)}")
        shutil.rmtree(project_dir)
        return
    

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    """
    модель ALINKED для keypoints и descriptors
    """
    aliked = ALIKED(max_num_keypoints=(4096*4), detection_threshold=0.3, resize=512).eval().to(device) # чуть чуть размоет детали у 300x300
    """
    KF.LightGlueMatcher — это компонент библиотеки Kornia,
      использующий модель LightGlue для сопоставления признаков.
    """
    matcher = KF.LightGlueMatcher(
        "aliked",
        {
            "width_confidence": 0.9,
            "depth_confidence": 0.9,
            "mp": True if 'cuda' in str(device) else False
        }
    ).eval().to(device)
    
    # создание директории для признаков
    feature_dir = os.path.join(project_dir, "features")
    os.makedirs(feature_dir, exist_ok=True)

    # Удаление существующих H5-файлов
    for h5_file in ['keypoints.h5', 'descriptors.h5', 'matches.h5']:
        h5_path = os.path.join(feature_dir, h5_file)
        if os.path.exists(h5_path):
            os.remove(h5_path)
            
    # Извлечение признаков с помощью ALIKED и сохранение в H5-файлы
    with h5py.File(os.path.join(feature_dir, 'keypoints.h5'), 'w') as f_kp, \
         h5py.File(os.path.join(feature_dir, 'descriptors.h5'), 'w') as f_desc:
        for image_name in image_names:
            with torch.inference_mode():
                img = load_image(os.path.join(project_dir, image_name)).to(device)
                feats = aliked.extract(img)
                """
                размер батча(1) | количество ключевых точек | координаты ключевых точек (x, y) 
                дескрипторы аналогично
                """
                kpts = feats['keypoints'].squeeze().cpu().numpy()
                descs = feats['descriptors'].squeeze().detach().cpu().numpy()
                f_kp[image_name] = kpts
                f_desc[image_name] = descs


    # загрузка в словари
    keypoints = {}
    descriptors = {}
    with h5py.File(os.path.join(feature_dir, 'keypoints.h5'), 'r') as f_kp, \
         h5py.File(os.path.join(feature_dir, 'descriptors.h5'), 'r') as f_desc:
        for image_name in image_names:
            keypoints[image_name] = f_kp[image_name][...]
            descriptors[image_name] = f_desc[image_name][...]
            
    # Сопоставление признаков с помощью LightGlue и сохранение в H5-файл
    with h5py.File(os.path.join(feature_dir, 'matches.h5'), 'w') as f_match:
        # pairs - это из дино матричка лигх глю ток с ними работает, далекие исключаем
        for idx1, idx2 in pairs:
            image_name1 = image_names[indices.index(idx1)]
            image_name2 = image_names[indices.index(idx2)]
            kp1 = torch.from_numpy(keypoints[image_name1]).to(device)
            kp2 = torch.from_numpy(keypoints[image_name2]).to(device)
            desc1 = torch.from_numpy(descriptors[image_name1]).to(device)
            desc2 = torch.from_numpy(descriptors[image_name2]).to(device)
            """
            Преобразует ключевые точки (kp1, kp2) в формат Local Affine Frames (LAF)
            kp1 и kp2 — это массивы NumPy формы (N1, 2) и (N2, 2)
            Nx - число ключевых точек x изображения.
            """
            laf1 = KF.laf_from_center_scale_ori(kp1[None])
            laf2 = KF.laf_from_center_scale_ori(kp2[None])
            with torch.inference_mode(): # отключение градиентов
                """
                LightGlue находит пары ключевых точек, чьи дескрипторы наиболее похожи, с учётом их локальной геометрии.
                """
                _, idxs = matcher(desc1, desc2, laf1, laf2) 
            if len(idxs) >= MIN_MATCHES:
                matches = idxs.cpu().numpy().astype(np.uint32) # (M,2) - [точка с инд i из 1 - соотв точке j из 2]
                """
                Группы в matches.h5 организуют соответствия по первому изображению. 
                """
                group = f_match.require_group(image_name1)
                group.create_dataset(image_name2, data=matches)


    with h5py.File(os.path.join(feature_dir, 'matches.h5'), 'r') as f_match:
        for g in f_match.keys():
            print(f"Группа: {g}, датасеты: {list(f_match[g].keys())}")
            for ds in f_match[g].keys():
                print(f"  {ds}: {f_match[g][ds][...].shape} совпадений")



    db_name = db_path
    db = COLMAPDatabase.connect(db_name)
    db.create_tables()
    """
    загрузка всего в database.db:
    - images
    - features (keypoints, descriptors)
    - модель камеры simple-pinhole (упрощённая модель без искажений).
    - False -> без масок
    """
    # получаем словарь (frame: id)
    fname_to_id = add_keypoints(db, feature_dir, project_dir, '', 'simple-pinhole', False)
    # добавляем матчи по image_id
    add_matches(db, feature_dir,fname_to_id)
    db.commit()


"""
сопоставление признаков (не только по pairs)
"""
    pycolmap.match_exhaustive(db_name, sift_options={'num_threads':1})
    # Реконструкция сцены
    """
    Результат: maps — словарь,
      где ключ — индекс модели (кластера), 
      а значение — объект Reconstruction, содержащий данные о камерах, изображениях и 3D-точках.
    """
    maps = pycolmap.incremental_mapping(
        database_path=db_path,
        image_path=project_dir,
        output_path='/kaggle/working/',
        options=pycolmap.IncrementalPipelineOptions({'min_model_size':5, 'max_num_models':3, 'num_threads':1}) # мин кол-в изображений в модели, макс кол-во отдельных кластеров в реконструкции
    )
    
    # Извлечение кластеров
    clusters = []
    print(f"Результаты COLMAP: {len(maps)} моделей")
    reconstructed_images = set()
    if maps:
        print("maps check")
        for map_index, cur_map in maps.items():
            print(f"Модель {map_index}: {len(cur_map.images)} изображений")
            cluster = {
                'cluster_index': map_index,
                'images': []
            }
            for image_id, image in cur_map.images.items():
                image_name = image.name
                print(f"Изображение в модели: {image_name}")
                if image_name in image_names:
                    prediction_index = image_names.index(image_name)
                    row_idx = indices[prediction_index]
                    rotation = image.cam_from_world.rotation.matrix().flatten()
                    translation = image.cam_from_world.translation
                    cluster['images'].append({
                        'row_idx': row_idx,
                        'image_name': image_name,
                        'rotation': rotation,
                        'translation': translation
                    })
                    reconstructed_images.add(image_name)
            if cluster['images']:
                clusters.append(cluster)
    
    # Добавление выбросов
    for image_name in image_names:
        if image_name not in reconstructed_images:
            print(image_name, " is looser")
            prediction_index = image_names.index(image_name)
            row_idx = indices[prediction_index]
            clusters.append({
                'cluster_index': None,
                'images': [{
                    'row_idx': row_idx,
                    'image_name': image_name,
                    'rotation': None,
                    'translation': None
                }]
            })
    
    # Очистка
    shutil.rmtree(project_dir)
    
    return clusters
    
    

In [68]:
# Формирование submission.csv
def write_submission(clusters_list):
    submission_file = '/kaggle/working/submission.csv'
    array_to_str = lambda array: ';'.join([f"{x:.09f}" for x in array]) if array is not None else ';'.join(['nan'] * 9)
    none_to_str = lambda n: ';'.join(['nan'] * n)
    
    with open(submission_file, 'w') as f:
        f.write('image_id,dataset,scene,image,rotation_matrix,translation_vector\n')
        for clusters in clusters_list:
            for cluster in clusters:
                cluster_name = 'outliers' if cluster['cluster_index'] is None else f'cluster{cluster["cluster_index"]}'
                for img_data in cluster['images']:
                    row = submission.loc[img_data['row_idx']]
                    image_id = row['image_id']
                    dataset = row['dataset']
                    image_path = row['image']
                    rotation = array_to_str(img_data['rotation'])
                    translation = none_to_str(3) if img_data['translation'] is None else ';'.join([f"{x:.09f}" for x in img_data['translation']])
                    f.write(f'{image_id},{dataset},{cluster_name},{image_path},{rotation},{translation}\n')
    
    print(f"Submission file created: {submission_file}")
    with open(submission_file, 'r') as f:
        print("\nПервые строки submission.csv:")
        for _ in range(5):
            print(f.readline().strip())

In [69]:
all_clusters = []
groups = submission.groupby(['dataset', 'scene'])

os.environ['QT_QPA_PLATFORM'] = 'offscreen'

for (dataset, scene), group in groups:
    dataset_path = os.path.join(test_path, dataset)
    if not os.path.exists(dataset_path):
        continue
    
    # Заменяем group_images_by_size на get_image_pairs
    pairs = get_image_pairs(dataset_path, scene, group)
    if not pairs:
        continue
    
    # Создаем временную группу на основе всех изображений
    temp_group = group
    temp_indices = [idx for idx, _ in group.iterrows()]
    clusters = process_group(dataset, scene, temp_group, temp_indices, pairs)
    if clusters:
        all_clusters.append(clusters)

write_submission(all_clusters)

Loaded LightGlue model
Группа: another_et_another_et001.png, датасеты: ['another_et_another_et002.png', 'another_et_another_et003.png', 'another_et_another_et004.png', 'another_et_another_et005.png', 'another_et_another_et006.png', 'another_et_another_et007.png']
  another_et_another_et002.png: (596, 2) совпадений
  another_et_another_et003.png: (275, 2) совпадений
  another_et_another_et004.png: (402, 2) совпадений
  another_et_another_et005.png: (472, 2) совпадений
  another_et_another_et006.png: (216, 2) совпадений
  another_et_another_et007.png: (175, 2) совпадений
Группа: another_et_another_et002.png, датасеты: ['another_et_another_et003.png', 'another_et_another_et004.png', 'another_et_another_et005.png', 'another_et_another_et006.png', 'another_et_another_et007.png']
  another_et_another_et003.png: (272, 2) совпадений
  another_et_another_et004.png: (412, 2) совпадений
  another_et_another_et005.png: (406, 2) совпадений
  another_et_another_et006.png: (225, 2) совпадений
  anoth

 45%|████▌     | 54/120 [00:00<00:00, 4339.52it/s]
I20250526 14:34:03.326382 139419726108224 misc.cc:44] 
Feature matching
I20250526 14:34:03.326682 139419734500928 sift.cc:1432] Creating SIFT CPU feature matcher
I20250526 14:34:03.326828 139419726108224 pairing.cc:168] Generating exhaustive image pairs...
I20250526 14:34:03.326849 139419726108224 pairing.cc:201] Matching block [1/1, 1/1]
I20250526 14:34:03.618465 139419726108224 feature_matching.cc:46] in 0.292s
I20250526 14:34:03.619433 139419726108224 timer.cc:91] Elapsed time: 0.005 [minutes]
I20250526 14:34:03.623380 139421116572800 incremental_pipeline.cc:237] Loading database
I20250526 14:34:03.624352 139421116572800 database_cache.cc:66] Loading cameras...
I20250526 14:34:03.624401 139421116572800 database_cache.cc:76]  22 in 0.000s
I20250526 14:34:03.624413 139421116572800 database_cache.cc:84] Loading matches...
I20250526 14:34:03.624718 139421116572800 database_cache.cc:89]  54 in 0.000s
I20250526 14:34:03.624735 13942111657

Результаты COLMAP: 2 моделей
maps check
Модель 0: 9 изображений
Изображение в модели: et_et000.png
Изображение в модели: et_et001.png
Изображение в модели: et_et002.png
Изображение в модели: et_et003.png
Изображение в модели: et_et004.png
Изображение в модели: et_et005.png
Изображение в модели: et_et006.png
Изображение в модели: et_et007.png
Изображение в модели: et_et008.png
Модель 1: 10 изображений
Изображение в модели: another_et_another_et001.png
Изображение в модели: another_et_another_et002.png
Изображение в модели: another_et_another_et003.png
Изображение в модели: another_et_another_et004.png
Изображение в модели: another_et_another_et005.png
Изображение в модели: another_et_another_et006.png
Изображение в модели: another_et_another_et007.png
Изображение в модели: another_et_another_et008.png
Изображение в модели: another_et_another_et009.png
Изображение в модели: another_et_another_et010.png
outliers_out_et001.png  is looser
outliers_out_et002.png  is looser
outliers_out_et003

 31%|███▏      | 48/153 [00:00<00:00, 3974.70it/s]
I20250526 14:34:45.517127 139419742893632 misc.cc:44] 
Feature matching
I20250526 14:34:45.517556 139419734500928 sift.cc:1432] Creating SIFT CPU feature matcher
I20250526 14:34:45.517729 139419742893632 pairing.cc:168] Generating exhaustive image pairs...
I20250526 14:34:45.517752 139419742893632 pairing.cc:201] Matching block [1/2, 1/2]
I20250526 14:34:45.820696 139419742893632 feature_matching.cc:46] in 0.303s
I20250526 14:34:45.821404 139419742893632 pairing.cc:201] Matching block [1/2, 2/2]
I20250526 14:34:45.821435 139419742893632 feature_matching.cc:46] in 0.000s
I20250526 14:34:45.821446 139419742893632 pairing.cc:201] Matching block [2/2, 1/2]
I20250526 14:34:45.879516 139419742893632 feature_matching.cc:46] in 0.058s
I20250526 14:34:45.880038 139419742893632 pairing.cc:201] Matching block [2/2, 2/2]
I20250526 14:34:45.880058 139419742893632 feature_matching.cc:46] in 0.000s
I20250526 14:34:45.880067 139419742893632 timer.cc:9

Результаты COLMAP: 1 моделей
maps check
Модель 0: 5 изображений
Изображение в модели: stairs_split_2_1710453756762.png
Изображение в модели: stairs_split_1_1710453689727.png
Изображение в модели: stairs_split_2_1710453871430.png
Изображение в модели: stairs_split_2_1710453736752.png
Изображение в модели: stairs_split_2_1710453739354.png
stairs_split_1_1710453576271.png  is looser
stairs_split_1_1710453601885.png  is looser
stairs_split_1_1710453606287.png  is looser
stairs_split_1_1710453612890.png  is looser
stairs_split_1_1710453616892.png  is looser
stairs_split_1_1710453620694.png  is looser
stairs_split_1_1710453626698.png  is looser
stairs_split_1_1710453643106.png  is looser
stairs_split_1_1710453651110.png  is looser
stairs_split_1_1710453659313.png  is looser
stairs_split_1_1710453663515.png  is looser
stairs_split_1_1710453667117.png  is looser
stairs_split_1_1710453668718.png  is looser
stairs_split_1_1710453675921.png  is looser
stairs_split_1_1710453678922.png  is looser
s

I20250526 14:34:46.503093 139421116572800 incremental_pipeline.cc:286] => No good initial image pair found.
I20250526 14:34:46.503396 139421116572800 incremental_pipeline.cc:282] Finding good initial image pair
I20250526 14:34:46.504427 139421116572800 incremental_pipeline.cc:286] => No good initial image pair found.
I20250526 14:34:46.504734 139421116572800 incremental_pipeline.cc:282] Finding good initial image pair
I20250526 14:34:46.505719 139421116572800 incremental_pipeline.cc:286] => No good initial image pair found.
I20250526 14:34:46.505986 139421116572800 incremental_pipeline.cc:282] Finding good initial image pair
I20250526 14:34:46.506787 139421116572800 incremental_pipeline.cc:286] => No good initial image pair found.
I20250526 14:34:46.507024 139421116572800 incremental_pipeline.cc:282] Finding good initial image pair
I20250526 14:34:46.507819 139421116572800 incremental_pipeline.cc:286] => No good initial image pair found.
I20250526 14:34:46.508061 139421116572800 increm